# NimbusPay ticket triage — LoRA fine-tune of Qwen2.5-1.5B-Instruct

Runs top to bottom on a free Colab **T4**. It audits `train.jsonl`, measures two baselines, runs five
controlled fine-tuning experiments, analyses errors on dev and writes `adapter/` + `predictions_test.jsonl`.

**Stack:** plain PyTorch training loop + Hugging Face `transformers` + `peft` (+ `bitsandbytes` for the 4-bit run).
I chose a hand-written loop over Unsloth/TRL because the whole pipeline (chat template, label masking, batching)
is ~40 visible lines that I can explain and change live, and it has no version-sensitive trainer wrappers.

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct` (1.54B params, Apache-2.0, not gated, has a system role in its chat
template, multilingual so Hinglish is not out of distribution). Small enough that 16-bit LoRA *and* 4-bit QLoRA
both fit a T4 and every experiment takes minutes.

In [ ]:
!pip -q install bitsandbytes peft
!pip -q uninstall -y torchao   # Colab ships an old torchao that peft refuses to import next to; not used here
import os, json, zipfile, shutil, time, gc, random, math, collections
import torch
print(torch.__version__, torch.cuda.get_device_name(0))

2.14.1+cu130 Tesla T4


## 0. Data
Put `training_assessment.zip` (the candidate pack) next to the notebook, or upload it when asked.

In [ ]:
FRESH = True     # True: ignore results cached by an earlier (interrupted) session and recompute everything
if FRESH:
    shutil.rmtree("runs", ignore_errors=True)
    if os.path.exists("results.json"): os.remove("results.json")
if not os.path.exists("data/train.jsonl"):
    if not os.path.exists("training_assessment.zip"):
        from google.colab import files
        files.upload()                      # choose training_assessment.zip
    zipfile.ZipFile("training_assessment.zip").extractall("pack")
    shutil.copytree("pack/candidate_pack/data", "data", dirs_exist_ok=True)
    for f in ("score.py", "SCHEMA.md"):
        shutil.copy(f"pack/candidate_pack/{f}", f)

def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

raw_train, dev, test = load("data/train.jsonl"), load("data/dev.jsonl"), load("data/test_inputs.jsonl")
print(len(raw_train), len(dev), len(test))

1445 200 400


## Part A — Audit of `train.jsonl`

`rules.py` is SCHEMA.md written as code: given a ticket and its category it derives every other field.
`audit.py` runs structural checks and then compares each label with what the spec says. Anything that disagrees
is either repaired (when the correct value is unambiguous) or dropped. Every change goes to `cleaning_log.csv`.

The rule code is used **only** on training data. It is never applied to model outputs.

In [ ]:
%%writefile rules.py
"""SCHEMA.md written as code.

Given a ticket and its category, derive every other field of the triage record.
Used ONLY to audit / repair train.jsonl and to label augmented training rows.
It is never used at inference time (the assignment forbids post-processing).
"""
import re
from datetime import date, timedelta

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
TXN_CATEGORIES = {"payment_failed", "refund", "fraud", "offers"}
MONTHS = {m: i + 1 for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}
MONTH_RE = (r"(jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|"
            r"sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)")

LEGAL_RE = re.compile(r"\b(rbi|ombudsman|consumer court|court|lawyer|legal notice|legal action|legal)\b", re.I)
REPEAT_RE = re.compile(
    r"third time|second complaint|again and again|already written|contacted support about this before|"
    r"baar baar|teesri baar|dusri baar|pehle bhi complaint|followed up|\b\d+(st|nd|rd|th) time\b", re.I)
# sentences whose rupee amounts are distractors (balance, limits, fees)
DISTRACTOR_RE = re.compile(r"balance|limit|\bfee\b|\bfees\b|pade hain", re.I)
HINGLISH = set("""hai hain hoon hua hui tha thi karo karna karun kiya kiye raha rahi rahe nahi nahin
mera meri mere mujhe maine mein ka ki aur lekin bhai kya kyun kaise abhi bhi gaya gayi gaye
paise paisa rupaye wapas chahiye baar pehle parso aaj yeh woh wala wali dhanyavaad namaste jaldi
kat liye liya diya dunga jaunga bhool badalne baat rakam poore mamla likha chuka kuch""".split())


def split_ticket(user_text):
    """-> (received_date, customer_message_without_quoted_lines)"""
    m = re.match(r"Received: (\d{4})-(\d{2})-(\d{2})", user_text)
    received = date(int(m[1]), int(m[2]), int(m[3]))
    body = user_text.split("\n\n", 1)[1] if "\n\n" in user_text else ""
    body = "\n".join(l for l in body.split("\n") if not l.lstrip().startswith(">"))
    return received, body


def txn_id(body):
    m = re.search(r"\bN\s?P[\s\-]*((?:\d[\s\-]?){10})(?!\d)", body, re.I)
    return "NP" + re.sub(r"\D", "", m[1]) if m else None


def _strip_noise(body):
    """Remove things that contain digits but are not amounts or dates."""
    b = re.sub(r"\bN\s?P[\s\-]*(?:\d[\s\-]?){10}", " ", body, flags=re.I)       # txn ids
    b = re.sub(r"#\d+|\b\d{9,}\b", " ", b)                                       # ticket / bank refs
    b = re.sub(r"(order number|order no\.?|order id)\s*\S+", " ", b, flags=re.I)
    b = re.sub(r"version (is )?\d+(\.\d+)+|\b\d+\.\d+\.\d+\b", " ", b, flags=re.I)  # app versions
    b = re.sub(r"(iPhone|Redmi Note|Realme|Samsung M|Vivo Y|Pixel|OnePlus)\s?\d+\w*", " ", b)
    return b


def _num(s):
    return float(s.replace(",", ""))


def amount_paise(body):
    b = _strip_noise(body)
    # drop distractor sentences; "Rs." must not count as a sentence end
    b = re.sub(r"\bRs\.", "Rs", b)
    # distractor clauses sometimes follow the real amount without a full stop
    b = re.sub(r"(FYI|My account balance|Account mein|Meri daily|A convenience fee)", r". \1", b)
    sents = [s for s in re.split(r"(?<=[.!?])\s+|\n+", b) if not DISTRACTOR_RE.search(s)]
    b = " ".join(sents)
    b = _DATE_STRIP.sub(" ", b)
    num = r"(\d[\d,]*(?:\.\d+)?)"
    cands = []
    for m in re.finditer(num + r"\s?(k|lakhs?|lacs?)\b", b, re.I):
        cands.append((m.start(), _num(m[1]) * (1000 if m[2].lower() == "k" else 100000)))
    if not cands:
        for m in re.finditer(r"(?:₹|\bRs\b\.?|\bINR\b)\s*" + num, b, re.I):
            cands.append((m.start(), _num(m[1])))
    if not cands:
        for m in re.finditer(num + r"\s?(?:rupees|rupaye|rs\b|/-)", b, re.I):
            cands.append((m.start(), _num(m[1])))
    if not cands:  # bare number with an amount cue
        for m in re.finditer(num + r"(?= (?:is the amount|was the value|in total|kat gaye|tha\b|ka tha|ka transaction))", b, re.I):
            cands.append((m.start(), _num(m[1])))
        for m in re.finditer(r"(?:amount|total|rakam|value)\D{0,12}" + num, b, re.I):
            cands.append((m.start(), _num(m[1])))
    if not cands:
        return None
    return int(round(cands[0][1] * 100))


_DATE_PATTERNS = [
    r"\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b",
    r"\b\d{1,2}(?:st|nd|rd|th)?\s+" + MONTH_RE + r"\b(?:\s+\d{4})?",
    r"\b" + MONTH_RE + r"\s+\d{1,2}(?:st|nd|rd|th)?\b(?:,?\s+\d{4})?",
]
_DATE_STRIP = re.compile("|".join(_DATE_PATTERNS) + r"|\b\d+\s+(?:days? ago|din pehle)", re.I)


def _most_recent(received, month, day):
    for y in (received.year, received.year - 1):
        try:
            d = date(y, month, day)
        except ValueError:
            continue
        if d <= received:
            return d
    return None


def txn_date(body, received):
    b = _strip_noise(body)
    low = b.lower()
    m = re.search(r"\b(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})\b", b)
    if m:
        y = int(m[3]); y += 2000 if y < 100 else 0
        return date(y, int(m[2]), int(m[1]))
    m = re.search(r"\b(\d{1,2})(?:st|nd|rd|th)?\s+" + MONTH_RE + r"\b(?:\s+(\d{4}))?", b, re.I)
    if not m:
        m2 = re.search(r"\b" + MONTH_RE + r"\s+(\d{1,2})(?:st|nd|rd|th)?\b(?:,?\s+(\d{4}))?", b, re.I)
        if m2:
            m = type("M", (), {"g": (m2[2], m2[1], m2[3])})
            day, mon, yr = m.g
        else:
            day = None
    else:
        day, mon, yr = m[1], m[2], m[3]
    if day:
        mon = MONTHS[mon[:3].lower()]
        return date(int(yr), mon, int(day)) if yr else _most_recent(received, mon, int(day))
    m = re.search(r"\b(\d+)\s+(?:days? ago|din pehle)", low)
    if m:
        return received - timedelta(days=int(m[1]))
    if re.search(r"day before yesterday|\bparso\b", low):
        return received - timedelta(days=2)
    if re.search(r"\byesterday\b|\bkal\b|last night", low):
        return received - timedelta(days=1)
    if re.search(r"\btoday\b|\baaj\b|this morning", low):
        return received
    m = re.search(r"last month on the (\d{1,2})", low)
    if m:
        first = received.replace(day=1) - timedelta(days=1)
        return first.replace(day=int(m[1]))
    m = re.search(r"\b(\d{1,2})(?:st|nd|rd|th) of this month", low)
    if m:
        return received.replace(day=int(m[1]))
    return None


def channel(body):
    low = body.lower()
    hits = []
    for name, pat in (("upi", r"\bupi\b"),
                      ("netbanking", r"net[\s-]?banking|internet banking"),
                      ("wallet", r"\bwallet\b"),
                      ("card", r"\bcard\b|\bvisa\b|mastercard")):
        m = re.search(pat, low)
        if m:
            hits.append((m.start(), name))
    # "scratch card" / "card details" are not a payment mode
    low2 = re.sub(r"\bs[a-z]{5,6} card\b|gift card", " ", low)   # "scratch card" (incl. typos)
    hits = [(p, n) for p, n in hits if n != "card" or re.search(r"\bcard\b|\bvisa\b|mastercard", low2)]
    return sorted(hits)[-1][1] if hits else None   # explicit "Mode:" sentences come after the issue sentence


def language(body):
    words = re.findall(r"[a-z]+", body.lower())
    return "hi-en" if sum(w in HINGLISH for w in words) >= 2 else "en"


def derive(user_text, category):
    received, body = split_ticket(user_text)
    rec = {"category": category, "priority": None, "amount_paise": None, "txn_date": None,
           "txn_id": None, "channel": None, "language": language(body), "needs_human": None}
    if category in TXN_CATEGORIES:
        rec["amount_paise"] = amount_paise(body)
        d = txn_date(body, received)
        rec["txn_date"] = d.isoformat() if d else None
        rec["txn_id"] = txn_id(body)
        rec["channel"] = channel(body)
    return finalize(rec, body)


def finalize(rec, body):
    """Fill priority and needs_human from category, amount and the ticket text."""
    category, amt = rec["category"], rec["amount_paise"] or 0
    if category == "fraud" or amt >= 50_000_00 or LEGAL_RE.search(body):
        rec["priority"] = "P1"
    elif (category in ("payment_failed", "refund") and amt >= 5_000_00) or category == "account_access":
        rec["priority"] = "P2"
    else:
        rec["priority"] = "P3"
    rec["needs_human"] = rec["priority"] == "P1" or bool(REPEAT_RE.search(body))
    return rec

Overwriting rules.py


In [ ]:
%%writefile audit.py
"""Part A: audit train.jsonl against SCHEMA.md, write the cleaned set and cleaning_log.csv.

    python audit.py            # reads data/train.jsonl (+ dev/test inputs, for leak check only)

Outputs: data/train_clean.jsonl, cleaning_log.csv, and a summary on stdout.
Every check is a few lines; the business rules live in rules.py.
"""
import collections
import csv
import json
import re

import rules


def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def norm_text(user_text):
    """Ticket text without header, case, spacing and the trailing 'Thx' some copies carry."""
    body = user_text.split("\n\n", 1)[-1].lower()
    body = re.sub(r"\s+", " ", body).strip()
    return re.sub(r"\s*thx$", "", body)


def main():
    train = load("data/train.jsonl")
    heldout = {norm_text(r["messages"][1]["content"])
               for f in ("data/dev.jsonl", "data/test_inputs.jsonl") for r in load(f)}
    log, clean, seen = [], [], {}
    stats = collections.Counter()

    for r in train:
        user, raw = r["messages"][1]["content"], r["messages"][2]["content"]
        _, body = rules.split_ticket(user)

        def drop(reason, key):
            log.append((r["id"], "drop", reason)); stats[key] += 1

        # 1. junk tickets ("test", "asdf asdf", empty) carrying a confident label
        if len(body.strip()) < 15:
            drop(f"junk ticket text {body.strip()!r} with a made-up label", "drop: junk text"); continue
        # 2. label is not valid JSON (cut off mid-string)
        try:
            gold = json.loads(raw)
        except json.JSONDecodeError:
            drop("label is truncated / invalid JSON", "drop: truncated label"); continue
        # 3. same ticket appears in dev/test -> leakage
        if norm_text(user) in heldout:
            drop("ticket text also appears in dev.jsonl (leak)", "drop: dev leak"); continue

        fixes = []
        # 4. legacy (v1) schema: key "prio" with low/medium/high, dates as DD/MM/YYYY
        if "prio" in gold:
            gold = {("priority" if k == "prio" else k): v for k, v in gold.items()}
            gold["priority"] = {"low": "P3", "medium": "P2", "high": "P1"}[gold["priority"]]
            fixes.append("legacy v1 label: prio low/medium/high -> priority P3/P2/P1"); stats["fix: legacy prio key"] += 1
        d = gold["txn_date"]
        if d and re.fullmatch(r"\d{2}/\d{2}/\d{4}", d):
            gold["txn_date"] = f"{d[6:]}-{d[3:5]}-{d[:2]}"
            fixes.append("txn_date DD/MM/YYYY -> YYYY-MM-DD"); stats["fix: legacy date format"] += 1
        if list(gold) != rules.KEYS:
            drop(f"unexpected keys {list(gold)}", "drop: bad keys"); continue

        # 5-7. compare every rule-derivable field with the spec (category is taken as given)
        want = rules.derive(user, gold["category"])
        if gold["txn_id"] != want["txn_id"] and gold["txn_id"] and re.sub(r"[^A-Z0-9]", "", gold["txn_id"].upper()) == want["txn_id"]:
            fixes.append(f"txn_id {gold['txn_id']!r} not normalised -> {want['txn_id']}"); stats["fix: txn_id not normalised"] += 1
            gold["txn_id"] = want["txn_id"]
        a, w = gold["amount_paise"], want["amount_paise"]
        if a != w and a is not None and w is not None and w // 100 == a:
            fixes.append(f"amount_paise {a} is in rupees -> {w} paise"); stats["fix: amount in rupees"] += 1
            gold["amount_paise"] = w
        if gold["priority"] != want["priority"]:
            fixes.append(f"priority {gold['priority']} contradicts spec -> {want['priority']}"); stats["fix: wrong priority"] += 1
            gold["priority"] = want["priority"]
        if gold["needs_human"] != want["needs_human"]:
            fixes.append(f"needs_human {gold['needs_human']} contradicts spec -> {want['needs_human']}"); stats["fix: wrong needs_human"] += 1
            gold["needs_human"] = want["needs_human"]
        # anything the rules still disagree with is something I could not explain -> drop, do not guess
        left = [k for k in rules.KEYS if gold[k] != want[k]]
        if left:
            drop(f"label disagrees with spec on {left} and could not be repaired safely", "drop: unexplained"); continue

        # 8. duplicate tickets (exact, or differing only in spacing / trailing 'Thx')
        key = norm_text(user)
        if key in seen:
            drop(f"duplicate of {seen[key]}", "drop: duplicate"); continue
        seen[key] = r["id"]

        if fixes:
            log.append((r["id"], "fix", "; ".join(fixes)))
        msgs = r["messages"][:2] + [{"role": "assistant", "content": json.dumps(gold, ensure_ascii=False)}]
        clean.append({"id": r["id"], "messages": msgs})

    with open("data/train_clean.jsonl", "w", encoding="utf-8") as f:
        for r in clean:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open("cleaning_log.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f); w.writerow(["id", "action", "reason"]); w.writerows(log)

    print(f"train rows: {len(train)}  ->  kept: {len(clean)}")
    print(f"rows dropped: {sum(a == 'drop' for _, a, _ in log)}   rows fixed: {sum(a == 'fix' for _, a, _ in log)}")
    for k, v in sorted(stats.items()):
        print(f"  {k:32} {v}")
    cats = collections.Counter(json.loads(r["messages"][2]["content"])["category"] for r in clean)
    print("category balance after cleaning:", dict(cats.most_common()))


if __name__ == "__main__":
    main()

Overwriting audit.py


In [ ]:
!python audit.py

train rows: 1445  ->  kept: 1274
rows dropped: 171   rows fixed: 175
  drop: dev leak                   45
  drop: duplicate                  91
  drop: junk text                  10
  drop: truncated label            25
  fix: amount in rupees            25
  fix: legacy date format          30
  fix: legacy prio key             70
  fix: txn_id not normalised       20
  fix: wrong priority              60
category balance after cleaning: {'payment_failed': 374, 'refund': 282, 'account_access': 164, 'kyc': 155, 'offers': 134, 'other': 90, 'fraud': 75}


In [ ]:
# Evidence: one raw example per problem type
import csv, re
log = list(csv.DictReader(open("cleaning_log.csv", encoding="utf-8")))
by_id = {r["id"]: r for r in raw_train}
seen = set()
for row in log:
    kind = re.sub(r"[\d'\"].*", "", row["reason"])[:28]
    if kind in seen: continue
    seen.add(kind)
    r = by_id[row["id"]]
    print(f"[{row['action']}] {row['id']}: {row['reason']}")
    print("   ticket:", r["messages"][1]["content"].split("\n\n", 1)[-1][:160].replace("\n", " | "))
    print("   label :", r["messages"][2]["content"][:190], "\n")

[drop] tr-00008: ticket text also appears in dev.jsonl (leak)
   ticket: Hi team, What documents are required for proof of identity?
   label : {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false} 

[fix] tr-00013: priority P1 contradicts spec -> P3
   ticket: Subject: Issue with my account |  | Hi, | KYC mein address update karna hai. Android app use kar raha hoon. Bank ref 782509120860 hai. Pehle bhi complaint kar chuka h
   label : {"category": "kyc", "priority": "P1", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "hi-en", "needs_human": true} 

[fix] tr-00020: legacy v1 label: prio low/medium/high -> priority P3/P2/P1; txn_date DD/MM/YYYY -> YYYY-MM-DD
   ticket: [IVR transcript] Hey, How can I check if I have received a refund if it is not yet on my statement? 978 rupees is the amount. I have contacted support about thi
   label : {"category": "r

In [ ]:
# How good is the category label? 10-fold cross-validated TF-IDF + logistic regression on the cleaned rows.
# Rows where an out-of-fold classifier confidently disagrees are candidates for label noise. I read them all:
# they are classifier mistakes (mostly fraud vs refund wording), not wrong labels, so category is left untouched.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
import numpy as np
clean = load("data/train_clean.jsonl")
X = [r["messages"][1]["content"].split("\n\n", 1)[-1].split("\n>")[0] for r in clean]
y = np.array([json.loads(r["messages"][2]["content"])["category"] for r in clean])
P = cross_val_predict(LogisticRegression(C=20, max_iter=2000), TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit_transform(X), y, cv=10, method="predict_proba")
pred = np.array(sorted(set(y)))[P.argmax(1)]
print(f"out-of-fold agreement with category labels: {(pred == y).mean():.1%}; disagreements: {(pred != y).sum()}")
for i in np.argsort(-P.max(1) * (pred != y))[:8]:
    print(f"  {P.max(1)[i]:.2f} label={y[i]:15} clf={pred[i]:15} {X[i][:110]!r}")

out-of-fold agreement with category labels: 94.7%; disagreements: 67
  0.87 label=refund          clf=payment_failed  'Hi, My card was declined but it shows as multiple charges? Rs. 2728 is the amount. It took place today.'
  0.82 label=fraud           clf=refund          "Subject: Help needed\n\nGood morning,\nI see a payment on my statement for a place I didn't make a purchase at. I"
  0.71 label=fraud           clf=payment_failed  'Subject: Complaint\n\n\nIt is urgency. Please freeze my card now. As someone in some odd remote town has withdraw'
  0.70 label=fraud           clf=refund          "@NimbusPayCare I don't recall making that payment? Using the Android app. Total ₹808. It was from the NimbusPa"
  0.69 label=fraud           clf=refund          "@NimbusPayCare I see a charge on my account that I don't recall making. I feel like my account may have been c"
  0.67 label=fraud           clf=payment_failed  '[IVR transcript] Hi, Payment I did not make. Dated Jun 6. Total ₹439. Mo

## Part C — Training pipeline

* **Chat template:** the model's own (`tokenizer.apply_chat_template`), with exactly the system + user message from the data.
* **Loss:** only on the assistant tokens (the JSON + `<|im_end|>`). Prompt tokens get label `-100`.
* **Sequence length:** chosen from the token-length evidence printed below.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
import score                                           # the provided scorer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 0
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"                              # only matters for batched generation
EOS = tok.convert_tokens_to_ids("<|im_end|>")

def prompt_text(row, system_extra=""):
    msgs = [dict(m) for m in row["messages"][:2]]
    msgs[0]["content"] += system_extra
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode(row):
    p = tok(prompt_text(row), add_special_tokens=False)["input_ids"]
    a = tok(row["messages"][2]["content"], add_special_tokens=False)["input_ids"] + [EOS]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

print(prompt_text(dev[0]) + dev[0]["messages"][2]["content"] + "<|im_end|>")

<|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-08-28 (Fri)
Source: app-chat

Good morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it.<|im_end|>
<|im_start|>assistant
{"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}<|im_end|>


In [ ]:
# Sequence-length evidence (prompt + label tokens)
def lens(rows): return np.array([len(encode(r)["input_ids"]) if len(r["messages"]) > 2 else len(tok(prompt_text(r))["input_ids"]) + 75 for r in rows])
for name, rows in (("train(clean)", clean), ("dev", dev), ("test (+75 for label)", test)):
    L = lens(rows)
    print(f"{name:22} median {int(np.median(L)):4d}  p95 {int(np.percentile(L, 95)):4d}  max {L.max():4d}   >512: {(L > 512).sum():3d}   >1024: {(L > 1024).sum():3d}   >2048: {(L > 2048).sum():2d}   >2304: {(L > 2304).sum()}")
MAX_LEN = 2304
# Emails carry long quoted threads AFTER the customer's text. The label is the last ~70 tokens of the sequence,
# so a 512 or 1024 cut-off would delete the label (train) or the question (inference cannot truncate at all).
# 2304 covers every row in train, dev and test, so nothing is ever truncated.
assert max(lens(clean).max(), lens(dev).max(), lens(test).max()) <= MAX_LEN

train(clean)           median  167  p95  224  max 2214   >512:  36   >1024:  36   >2048:  9   >2304: 0
dev                    median  172  p95  222  max 2180   >512:   3   >1024:   3   >2048:  2   >2304: 0
test (+75 for label)   median  183  p95  226  max 2164   >512:  11   >1024:  11   >2048:  1   >2304: 0


In [ ]:
def load_model(four_bit=False):
    kw = {"device_map": {"": 0}}
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    else:
        kw["torch_dtype"] = torch.float16
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)

@torch.no_grad()
def predict(model, rows, batch_size=16, system_extra=""):
    """Greedy decoding, max_new_tokens=200, output = generated text with whitespace stripped. Nothing else."""
    model.eval()
    texts = [prompt_text(r, system_extra) for r in rows]
    order = sorted(range(len(rows)), key=lambda i: -len(texts[i]))        # longest first, less padding
    out = [None] * len(rows)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, do_sample=False, max_new_tokens=200, temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id)
        for i, g in zip(idx, gen):
            out[i] = tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return [{"id": r["id"], "output": o} for r, o in zip(rows, out)]

def evaluate(preds, gold=dev):
    res = score.score(gold, preds)
    return res, {"exact": round(res["all"]["exact_match"], 4), "field_acc": round(res["all"]["mean_field_acc"], 4),
                 "json_valid": round(res["all"]["json_valid"], 4)}

RESULTS = json.load(open("results.json")) if os.path.exists("results.json") else {}
def save_results():
    json.dump(RESULTS, open("results.json", "w"), indent=1)

## Part B — Baselines (no fine-tuning)
1. base model, fixed prompt;  2. base model with the whole SCHEMA.md appended to the system prompt (reference only).

In [ ]:
if "baseline_fixed_prompt" not in RESULTS or "baseline_rules_in_prompt" not in RESULTS:
    base = load_model()
    rules_text = "\n\nFollow this labelling spec exactly. Reply with only the JSON object.\n\n" + open("SCHEMA.md", encoding="utf-8").read()
    for name, extra, bs in (("baseline_fixed_prompt", "", 16), ("baseline_rules_in_prompt", rules_text, 8)):
        t = time.time(); preds = predict(base, dev, bs, extra)
        _, RESULTS[name] = evaluate(preds); RESULTS[name]["sample"] = preds[0]["output"][:300]
        print(name, RESULTS[name], f"{(time.time() - t) / 60:.1f} min")
    save_results(); del base; gc.collect(); torch.cuda.empty_cache()
for k in ("baseline_fixed_prompt", "baseline_rules_in_prompt"): print(k, RESULTS[k])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

baseline_fixed_prompt {'exact': 0.0, 'field_acc': 0.0, 'json_valid': 0.0, 'sample': '```json\n{\n  "status": "New",\n  "priority": "High",\n  "category": "Card Activation",\n  "description": "User received an activation request for their card but has not yet been able to use it. They need assistance with activating and using the card.",\n  "ticket_number": "#54047"\n}\n```'} 1.8 min
baseline_rules_in_prompt {'exact': 0.005, 'field_acc': 0.3775, 'json_valid': 0.625, 'sample': '{\n  "category": "kyc",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "language": "en",\n  "needs_human": true\n}'} 3.3 min
baseline_fixed_prompt {'exact': 0.0, 'field_acc': 0.0, 'json_valid': 0.0, 'sample': '```json\n{\n  "status": "New",\n  "priority": "High",\n  "category": "Card Activation",\n  "description": "User received an activation request for their card but has not yet been able to use it. They need assistance with activating and using the c

## Training loop

Main settings and why:

| Setting | Value | Reason |
|---|---|---|
| LoRA targets | all attention + MLP projections | the task needs new *computation* (date arithmetic, paise conversion, priority rules), not just a new output style; MLP adapters help with that |
| rank / alpha | 16 / 32 | 18.5M trainable params (limit 50M); rank is one of the ablations |
| learning rate | 2e-4, cosine, 5% warm-up | standard LoRA rate; adapters start at zero so a high rate is safe |
| epochs | 3 | ~1.3k rows; loss is flat by epoch 3 (see log) |
| batching | up to 12 rows, at most 2,304 padded tokens per batch (3,000 ran out of memory on the T4) | most tickets are ~200 tokens but threaded emails are ~1,700; a token budget keeps peak memory flat without truncating anything |
| memory | gradient checkpointing + logits only at label positions | without these two, one 2.2k-token email alone overflows the T4's 15 GB |
| precision | fp16 autocast, LoRA weights in fp32 | T4 has no fast bf16 |

In [ ]:
def make_batches(data, rng, max_rows=12, max_tokens=2304):
    idx = list(range(len(data))); rng.shuffle(idx)
    batches = []
    for s in range(0, len(idx), 256):                       # sort inside chunks: similar lengths share a batch
        chunk = sorted(idx[s:s + 256], key=lambda i: len(data[i]["input_ids"]))
        cur = []
        for i in chunk:
            longest = len(data[i]["input_ids"])
            if cur and (len(cur) == max_rows or (len(cur) + 1) * longest > max_tokens):
                batches.append(cur); cur = []
            cur.append(i)
        if cur: batches.append(cur)
    rng.shuffle(batches)
    return batches

def collate(items):
    n = max(len(x["input_ids"]) for x in items)
    ids = torch.full((len(items), n), tok.pad_token_id); lab = torch.full((len(items), n), -100); att = torch.zeros((len(items), n), dtype=torch.long)
    for j, x in enumerate(items):
        k = len(x["input_ids"])
        ids[j, :k] = torch.tensor(x["input_ids"]); lab[j, :k] = torch.tensor(x["labels"]); att[j, :k] = 1
    return ids.cuda(), att.cuda(), lab.cuda()

def lm_loss(model, ids, att, lab):
    """Cross-entropy on the assistant tokens only. Logits are computed just at those positions:
    full logits (151k vocab x ~2k prompt tokens) for a threaded email do not fit a T4."""
    core = model.get_base_model()                        # Qwen2ForCausalLM with the LoRA layers injected
    h = core.model(input_ids=ids, attention_mask=att).last_hidden_state
    keep = lab[:, 1:] != -100                            # position t predicts token t+1
    logits = core.lm_head(h[:, :-1][keep])
    return torch.nn.functional.cross_entropy(logits.float(), lab[:, 1:][keep])

def run(name, rows, four_bit=False, r=16, lr=2e-4, epochs=3, note=""):
    if name in RESULTS:                                      # finished earlier in this session: reuse the saved log
        print(name, "(cached)", RESULTS[name]); return
    random.seed(SEED); torch.manual_seed(SEED); rng = random.Random(SEED)
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = load_model(four_bit)
    if four_bit: model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    # gradient checkpointing: a 2.2k-token threaded email needs ~10 GB of activations without it (OOM on a T4)
    model.gradient_checkpointing_enable(); model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, task_type="CAUSAL_LM",
                           target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    data = [encode(x) for x in rows]
    data = [x for x in data if len(x["input_ids"]) <= MAX_LEN]
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    steps = sum(len(make_batches(data, random.Random(e))) for e in range(epochs))
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    model.train(); t0 = time.time(); step = 0; losses = []
    for ep in range(epochs):
        for b in make_batches(data, rng):
            ids, att, lab = collate([data[i] for i in b])
            with torch.autocast("cuda", dtype=torch.float16):
                loss = lm_loss(model, ids, att, lab)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            losses.append(loss.item()); step += 1
            if step % 40 == 0: print(f"  {name} epoch {ep + 1} step {step}/{steps} loss {np.mean(losses[-40:]):.4f}")
    minutes = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30
    model.save_pretrained(f"runs/{name}")
    preds = predict(model, dev)
    json.dump(preds, open(f"runs/{name}/dev_preds.json", "w"))
    _, m = evaluate(preds)
    RESULTS[name] = dict(m, note=note, rows=len(data), trainable_params=trainable, peak_gpu_gb=round(peak, 2),
                         train_min=round(minutes, 1), final_loss=round(float(np.mean(losses[-20:])), 4))
    save_results(); print(name, RESULTS[name])
    del model, opt; gc.collect(); torch.cuda.empty_cache()

## Part D — Controlled experiments
`A` is the reference run. Every other run changes exactly one thing relative to `A`.

In [ ]:
%%writefile augment.py
"""Experiment 4 (own idea): rule-targeted augmentation.

The cleaned train set states almost every date / amount / id in a handful of surface forms.
SCHEMA.md asks for more than that (relative dates, missing years, k / lakh, ids with spaces and
symbols, distracting amounts and reference numbers, null transaction fields for non-transaction
categories). Each synthetic row = one cleaned train ticket + ONE inserted sentence whose effect
on the label is known by construction. Uses only train.jsonl and SCHEMA.md.
"""
import json
import random
import re
from datetime import date, timedelta

import rules

MON = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def _indian(n):
    """123456 -> '1,23,456' (Indian digit grouping)."""
    s = str(n)
    if len(s) <= 3:
        return s
    head, tail = s[:-3], s[-3:]
    parts = []
    while len(head) > 2:
        parts.insert(0, head[-2:]); head = head[:-2]
    return ",".join(([head] if head else []) + parts + [tail])


def _ord(n):
    return f"{n}{'th' if 10 < n % 100 < 14 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"


def _insert(user, sentence, rng):
    """Insert a sentence at a random sentence boundary of the customer's main paragraph."""
    head, body = user.split("\n\n", 1)
    lines = body.split("\n")
    cand = [i for i, l in enumerate(lines)
            if l.strip() and not l.startswith((">", "Subject:", "Regards")) and re.search(r"[.!?]", l)]
    i = max(cand, key=lambda j: len(lines[j]))
    sents = re.split(r"(?<=[.!?]) +", lines[i])
    k = rng.randint(1, len(sents))
    lines[i] = " ".join(sents[:k] + [sentence] + sents[k:])
    return head + "\n\n" + "\n".join(lines)


def _new_id(rng):
    d = "".join(rng.choice("0123456789") for _ in range(10))
    shown = rng.choice([f"N P {d[:4]} {d[4:7]} {d[7:]}", f"Np-{d[:5]}-{d[5:]}", f"np.{d}", f"NP#{d}",
                        f"nP {d[:2]} {d[2:6]} {d[6:]}", f"NP_{d[:5]}_{d[5:]}", f"np-{d[:3]}-{d[3:6]}-{d[6:]}"])
    return shown, "NP" + d


def op_id(user, gold, hi, rng):
    shown, norm = _new_id(rng)
    m = re.search(r"\bN\s?P[\s\-]*(?:\d[\s\-]?){10}", user, re.I)
    if gold["txn_id"] and m:                       # same ticket, id written in a new way
        return user[:m.start()] + shown + " " + user[m.end():].lstrip(), {"txn_id": norm}
    if gold["txn_id"] is None:
        s = rng.choice(["Iska transaction number {} hai.", "App mein id {} dikha raha hai."] if hi else
                       ["Payment reference: {}.", "The app shows id {} for it.", "NimbusPay id on the SMS is {}."])
        return _insert(user, s.format(shown), rng), {"txn_id": norm}


def op_date(user, gold, hi, rng):
    if gold["txn_date"] is not None:
        return None
    rec, _ = rules.split_ticket(user)
    kind = rng.choice(["night", "week", "ndays", "lastmonth", "thismonth", "noyear", "prevyear", "parso"])
    if kind == "night":
        d, s = rec - timedelta(1), ("Kal raat ko hua yeh." if hi else "This happened last night.")
    elif kind == "week":
        d, s = rec - timedelta(7), ("Ek hafta pehle hua tha." if hi else "It happened exactly a week ago.")
    elif kind == "ndays":
        n = rng.randint(2, 40)
        d, s = rec - timedelta(n), (f"{n} din pehle ki baat hai." if hi else f"That was {n} days back.")
    elif kind == "parso":
        d, s = rec - timedelta(2), ("Parso hua tha yeh." if hi else "It was the day before yesterday.")
    elif kind == "lastmonth":
        last = rec.replace(day=1) - timedelta(1)
        d = last.replace(day=rng.randint(1, last.day))
        s = f"Pichle mahine ki {d.day} tareekh ko hua." if hi else f"It was on the {_ord(d.day)} of last month."
    elif kind == "thismonth":
        d = rec.replace(day=rng.randint(1, rec.day))
        s = f"Isi mahine ki {d.day} tareekh ko hua." if hi else f"It happened on the {_ord(d.day)} of this month."
    else:  # day + month without a year -> most recent such date on or before Received
        back = rng.randint(3, 120) if kind == "noyear" else rng.randint(200, 350)
        d = rec - timedelta(back)
        form = rng.choice([f"{d.day} {MON[d.month - 1]}", f"{MON[d.month - 1]} {d.day}"])
        s = f"{form} ko hua tha." if hi else f"The date was {form}."
    return _insert(user, s, rng), {"txn_date": d.isoformat()}


def op_amount(user, gold, hi, rng):
    if gold["amount_paise"] is not None:
        return None
    kind = rng.choice(["slash", "space", "k", "lakh", "plain", "comma", "paise"])
    if kind == "k":
        x = rng.choice([rng.randint(1, 99), rng.randint(10, 990) / 10]); rupees, shown = x * 1000, f"{x}k"
    elif kind == "lakh":
        x = rng.choice([1, 2, 3, 1.5, 2.5, 0.5, 0.75, 1.2]); rupees, shown = x * 100000, f"{x} lakh"
    else:
        n = rng.choice([rng.randint(20, 999), rng.randint(1000, 9999), rng.randint(10000, 99999), rng.randint(100000, 400000)])
        rupees = n
        shown = {"slash": f"Rs.{n}/-", "space": f"₹ {n}.00", "plain": f"{n} rupees", "comma": f"INR {_indian(n)}"}.get(kind)
        if kind == "paise":
            p = rng.choice([25, 50, 75, 99]); rupees = n + p / 100; shown = f"Rs {_indian(n)}.{p}"
    s = rng.choice(["{} ka payment tha.", "Kul {} ka mamla hai."] if hi else
                   ["The sum in question is {}.", "I am out {} because of this.", "{} is what it was for."])
    return _insert(user, s.format(shown), rng), {"amount_paise": int(round(rupees * 100))}


def op_channel(user, gold, hi, rng):
    if gold["channel"] is not None:
        return None
    ch = rng.choice(["upi", "card", "netbanking", "wallet"])
    s = rng.choice({
        "upi": ["UPI app se bheja tha."] if hi else ["I paid using the UPI app.", "Payment mode was UPI."],
        "card": ["Visa debit card se pay kiya tha."] if hi else ["I used my Mastercard for it.", "It was paid by Visa.", "I paid by credit card."],
        "netbanking": ["Net-banking se kiya tha."] if hi else ["It was done from net-banking.", "I paid through internet banking."],
        "wallet": ["NimbusPay wallet se gaya tha."] if hi else ["It went out of my wallet.", "I paid from the wallet."],
    }[ch])
    return _insert(user, s, rng), {"channel": ch}


def op_distractor(user, gold, hi, rng):
    """Numbers that must be ignored: label does not change."""
    n = rng.randint(50, 90000)
    s = rng.choice(
        [f"Order number {rng.randint(10000, 99999)} hai.", f"Meri daily limit ₹{_indian(n)} hai.",
         f"Wallet mein abhi {n} rupaye bache hain, balance check kar lo.", f"Upar se Rs. {rng.randint(5, 99)} ki fee bhi lagi."] if hi else
        [f"Order number {rng.randint(10000, 99999)} if that helps.", f"My transfer limit is ₹{_indian(n)}.",
         f"My balance shows Rs. {n} at the moment.", f"A convenience fee of Rs. {rng.randint(5, 99)} was also added.",
         f"The bank reference number is {rng.randint(10**11, 10**12 - 1)}.", f"My old ticket was #{rng.randint(10000, 99999)}."])
    return _insert(user, s, rng), {}


OPS = [op_id, op_date, op_amount, op_channel, op_distractor]


def augment(clean_rows, n=500, seed=0):
    rng = random.Random(seed)
    out = []
    while len(out) < n:
        r = rng.choice(clean_rows)
        user = r["messages"][1]["content"]
        gold = json.loads(r["messages"][2]["content"])
        hi = gold["language"] == "hi-en"
        if gold["category"] in rules.TXN_CATEGORIES:
            res = rng.choice(OPS)(user, gold, hi, rng)
        else:
            # non-transaction categories: transaction details may be mentioned, fields stay null
            g2 = dict(gold, amount_paise=None, txn_date=None, txn_id=None, channel=None)
            res = rng.choice([op_id, op_date, op_channel, op_distractor])(user, g2, hi, rng)
            if res:
                res = (res[0], {})
        if not res:
            continue
        new_user, change = res
        gold.update(change)
        _, body = rules.split_ticket(new_user)
        gold = rules.finalize(gold, body)          # amount may change priority / needs_human
        out.append({"id": f"aug-{len(out):04d}-from-{r['id']}",
                    "messages": [r["messages"][0], {"role": "user", "content": new_user},
                                 {"role": "assistant", "content": json.dumps(gold, ensure_ascii=False)}]})
    return out


if __name__ == "__main__":
    clean = [json.loads(l) for l in open("data/train_clean.jsonl", encoding="utf-8")]
    rows = augment(clean)
    with open("data/train_aug.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(len(rows), "synthetic rows written to data/train_aug.jsonl")
    for r in rows[:6]:
        print("\n" + r["messages"][1]["content"].split("\n> ")[0], "\n ->", r["messages"][2]["content"])

Overwriting augment.py


In [ ]:
import augment
aug = augment.augment(clean, n=500, seed=0)
print(len(clean), "clean rows +", len(aug), "synthetic rows. Examples:")
for r in aug[:3]: print("\n" + r["messages"][1]["content"].split("\n\n", 1)[1].split("\n>")[0][:300], "\n ->", r["messages"][2]["content"])

1274 clean rows + 500 synthetic rows. Examples:

[IVR transcript] Bhai, KYC mein address update karna hai. Pichla ticket number #58745 tha. Please help karo. Wallet mein abhi 53125 rupaye bache hain, balance check kar lo. 
 -> {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "hi-en", "needs_human": false}

[IVR transcript] URGENT!! Says my transfer can't be completed? Transaction date 31/08/2026. The amount was 163612 rupees. Kindly resolve this asap. My balance shows Rs. 18304 at the moment. 
 -> {"category": "payment_failed", "priority": "P1", "amount_paise": 16361200, "txn_date": "2026-08-31", "txn_id": null, "channel": null, "language": "en", "needs_human": true}

what documents do i need to verify who i am. I am using app version 7.5.18. Please help. It happened on the 4th of this month. 
 -> {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "langu

In [ ]:
run("A_clean_r16_16bit", clean, note="reference: cleaned data, 16-bit LoRA, r=16, lr 2e-4, 3 epochs")
run("B_raw_data", raw_train, note="exp 1: raw train.jsonl instead of cleaned")
run("C_qlora_4bit", clean, four_bit=True, note="exp 2: base model in 4-bit NF4 (QLoRA) instead of 16-bit")
run("D_rank4", clean, r=4, note="exp 3: LoRA rank 4 instead of 16")
run("E_clean_plus_aug", clean + aug, note="exp 4 (own idea): + 500 rule-targeted synthetic rows")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  A_clean_r16_16bit epoch 1 step 40/437 loss 0.5472
  A_clean_r16_16bit epoch 1 step 80/437 loss 0.0423
  A_clean_r16_16bit epoch 1 step 120/437 loss 0.0324
  A_clean_r16_16bit epoch 2 step 160/437 loss 0.0195
  A_clean_r16_16bit epoch 2 step 200/437 loss 0.0095
  A_clean_r16_16bit epoch 2 step 240/437 loss 0.0081
  A_clean_r16_16bit epoch 2 step 280/437 loss 0.0059
  A_clean_r16_16bit epoch 3 step 320/437 loss 0.0040
  A_clean_r16_16bit epoch 3 step 360/437 loss 0.0030
  A_clean_r16_16bit epoch 3 step 400/437 loss 0.0011


A_clean_r16_16bit {'exact': 0.86, 'field_acc': 0.9731, 'json_valid': 1.0, 'note': 'reference: cleaned data, 16-bit LoRA, r=16, lr 2e-4, 3 epochs', 'rows': 1274, 'trainable_params': 18464768, 'peak_gpu_gb': 4.79, 'train_min': 12.6, 'final_loss': 0.0015}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  B_raw_data epoch 1 step 40/492 loss 0.5813
  B_raw_data epoch 1 step 80/492 loss 0.0775
  B_raw_data epoch 1 step 120/492 loss 0.0547
  B_raw_data epoch 1 step 160/492 loss 0.0500
  B_raw_data epoch 2 step 200/492 loss 0.0391
  B_raw_data epoch 2 step 240/492 loss 0.0339
  B_raw_data epoch 2 step 280/492 loss 0.0243
  B_raw_data epoch 2 step 320/492 loss 0.0180
  B_raw_data epoch 3 step 360/492 loss 0.0151
  B_raw_data epoch 3 step 400/492 loss 0.0128
  B_raw_data epoch 3 step 440/492 loss 0.0106
  B_raw_data epoch 3 step 480/492 loss 0.0149
B_raw_data {'exact': 0.825, 'field_acc': 0.9719, 'json_valid': 1.0, 'note': 'exp 1: raw train.jsonl instead of cleaned', 'rows': 1445, 'trainable_params': 18464768, 'peak_gpu_gb': 4.81, 'train_min': 14.3, 'final_loss': 0.0137}


... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  D_rank4 epoch 1 step 40/437 loss 0.8157
  D_rank4 epoch 1 step 80/437 loss 0.0543
  D_rank4 epoch 1 step 120/437 loss 0.0384
  D_rank4 epoch 2 step 160/437 loss 0.0279
  D_rank4 epoch 2 step 200/437 loss 0.0235
  D_rank4 epoch 2 step 240/437 loss 0.0144
  D_rank4 epoch 2 step 280/437 loss 0.0107
  D_rank4 epoch 3 step 320/437 loss 0.0057
  D_rank4 epoch 3 step 360/437 loss 0.0044
  D_rank4 epoch 3 step 400/437 loss 0.0025
D_rank4 {'exact': 0.815, 'field_acc': 0.965, 'json_valid': 1.0, 'note': 'exp 3: LoRA rank 4 instead of 16', 'rows': 1274, 'trainable_params': 4616192, 'peak_gpu_gb': 4.63, 'train_min': 12.7, 'final_loss': 0.004}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  E_clean_plus_aug epoch 1 step 40/598 loss 0.6339
  E_clean_plus_aug epoch 1 step 80/598 loss 0.0552
  E_clean_plus_aug epoch 1 step 120/598 loss 0.0362
  E_clean_plus_aug epoch 1 step 160/598 loss 0.0254
  E_clean_plus_aug epoch 2 step 200/598 loss 0.0159
  E_clean_plus_aug epoch 2 step 240/598 loss 0.0078
  E_clean_plus_aug epoch 2 step 280/598 loss 0.0058
  E_clean_plus_aug epoch 2 step 320/598 loss 0.0073
  E_clean_plus_aug epoch 2 step 360/598 loss 0.0042
  E_clean_plus_aug epoch 3 step 400/598 loss 0.0024
  E_clean_plus_aug epoch 3 step 440/598 loss 0.0014
  E_clean_plus_aug epoch 3 step 480/598 loss 0.0009
  E_clean_plus_aug epoch 3 step 520/598 loss 0.0010
  E_clean_plus_aug epoch 3 step 560/598 loss 0.0033
E_clean_plus_aug {'exact': 0.95, 'field_acc': 0.99, 'json_valid': 1.0, 'note': 'exp 4 (own idea): + 500 rule-targeted synthetic rows', 'rows': 1774, 'trainable_params': 18464768, 'peak_gpu_gb': 4.8, 'train_min': 17.3, 'final_loss': 0.001}


In [ ]:
import pandas as pd
tab = pd.DataFrame(RESULTS).T[["note", "exact", "field_acc", "json_valid", "peak_gpu_gb", "train_min", "rows", "trainable_params"]]
tab

,note,exact,field_acc,json_valid,peak_gpu_gb,train_min,rows,trainable_params
baseline_fixed_prompt,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN
baseline_rules_in_prompt,NaN,0.005,0.3775,0.625,NaN,NaN,NaN,NaN
A_clean_r16_16bit,"reference: cleaned data, 16-bit LoRA, r=16, lr...",0.86,0.9731,1.0,4.79,12.6,1274,18464768
B_raw_data,exp 1: raw train.jsonl instead of cleaned,0.825,0.9719,1.0,4.81,14.3,1445,18464768
C_qlora_4bit,exp 2: base model in 4-bit NF4 (QLoRA) instead...,0.87,0.9769,1.0,4.03,15.9,1274,18464768
D_rank4,exp 3: LoRA rank 4 instead of 16,0.815,0.965,1.0,4.63,12.7,1274,4616192
E_clean_plus_aug,exp 4 (own idea): + 500 rule-targeted syntheti...,0.95,0.99,1.0,4.8,17.3,1774,18464768


## Final model, Part E error analysis, test predictions

The final model is the run with the best dev exact match. It is reloaded **from the saved adapter** onto a fresh
16-bit base model and decoded **one ticket at a time** — the same way the graders will run it — for both the final
dev numbers and the test predictions.

In [ ]:
runs = [k for k in RESULTS if not k.startswith("baseline") and k != "FINAL"]
best = max(runs, key=lambda k: (RESULTS[k]["exact"], RESULTS[k]["field_acc"]))
print("best run:", best)
shutil.rmtree("adapter", ignore_errors=True); shutil.copytree(f"runs/{best}", "adapter", ignore=shutil.ignore_patterns("dev_preds.json"))
final = PeftModel.from_pretrained(load_model(four_bit=(best == "C_qlora_4bit")), "adapter")
dev_preds = predict(final, dev, batch_size=1)
res, m = evaluate(dev_preds)
RESULTS["FINAL"] = dict(m, note=f"{best} reloaded from adapter/, batch size 1"); save_results()
with open("predictions_dev.jsonl", "w", encoding="utf-8") as f:
    for p in dev_preds: f.write(json.dumps(p, ensure_ascii=False) + "\n")
!python score.py --gold data/dev.jsonl --pred predictions_dev.jsonl

best run: E_clean_plus_aug


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

group                            n   valid   exact  fields
all                            200  100.0%   95.0%   99.0%
category=account_access         19  100.0%  100.0%  100.0%
category=fraud                  15  100.0%   86.7%   95.8%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%  100.0%  100.0%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   96.2%   99.5%
category=refund                 55  100.0%   89.1%   98.0%

per-field accuracy (all rows)
  category        98.5%
  priority        96.0%
  amount_paise    99.5%
  txn_date       100.0%
  txn_id         100.0%
  channel        100.0%
  language       100.0%
  needs_human     98.0%


In [ ]:
# Error analysis: every dev failure, with the fields that are wrong
gold = {r["id"]: json.loads(r["messages"][2]["content"]) for r in dev}
text = {r["id"]: r["messages"][1]["content"] for r in dev}
fails, field_err = [], collections.Counter()
for p in dev_preds:
    g = gold[p["id"]]; o = score.parse(p["output"]) or {}
    wrong = [k for k in score.KEYS if not (k in o and score.same(o[k], g[k]))]
    if wrong: fails.append((p["id"], wrong, o)); field_err.update(wrong)
print(len(fails), "of", len(dev), "dev tickets wrong. Errors per field:", dict(field_err.most_common()))
thread = {i for i, t in text.items() if "\n>" in t}
for name, ids in (("email with quoted thread", thread), ("Hinglish", {i for i, g in gold.items() if g["language"] == "hi-en"}), ("English", {i for i, g in gold.items() if g["language"] == "en"})):
    bad = sum(f[0] in ids for f in fails); print(f"  slice {name:26} n={len(ids):3d} exact={1 - bad / max(len(ids), 1):.1%}")
for i, wrong, o in fails[:25]:
    print("\n" + "=" * 100 + f"\n{i}  wrong: " + ", ".join(f"{k}: pred={o.get(k)!r} gold={gold[i][k]!r}" for k in wrong))
    print(text[i].split("\n>")[0].strip()[:600])

10 of 200 dev tickets wrong. Errors per field: {'priority': 8, 'needs_human': 4, 'category': 3, 'amount_paise': 1}
  slice email with quoted thread   n=  3 exact=66.7%
  slice Hinglish                   n= 49 exact=98.0%
  slice English                    n=151 exact=94.0%

dv-00011  wrong: priority: pred='P2' gold='P1', needs_human: pred=False gold=True
Received: 2026-06-17 (Wed)
Source: email

Subject: Problem with NimbusPay

URGENT!!
Why am I missing my refund. App version is 4.2.7. ₹84,137 was the value. This happened yesterday. ID: np3333133441

Regards,
Pooja Nair

dv-00017  wrong: amount_paise: pred=18082525 gold=18082725
Received: 2026-06-30 (Tue)
Source: twitter

@NimbusPayCare I am still waiting for a refund. Mode: internet banking. My account balance is 712 rupees right now. It took place on 13th June 2026. You can check np5682637619. The amount was Rs 1,80,827.25.

dv-00037  wrong: priority: pred='P2' gold='P1', needs_human: pred=False gold=True
Received: 2026-09-07 (Mon)
S

In [ ]:
t = time.time()
test_preds = predict(final, test, batch_size=1)
with open("predictions_test.jsonl", "w", encoding="utf-8") as f:
    for p in test_preds: f.write(json.dumps(p, ensure_ascii=False) + "\n")
valid = sum(score.parse(p["output"]) is not None and list(score.parse(p["output"])) == score.KEYS for p in test_preds)
print(len(test_preds), "test predictions,", valid, "valid JSON with the 8 keys,", f"{(time.time() - t) / 60:.1f} min")
print(test_preds[0])

400 test predictions, 399 valid JSON with the 8 keys, 33.8 min
{'id': 'ts-00001', 'output': '{"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-06-30", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}'}


In [ ]:
# Reproducibility self-check: fresh base model + adapter/ from disk, 20 random test rows, must match the file exactly
del final; gc.collect(); torch.cuda.empty_cache()
check = PeftModel.from_pretrained(load_model(four_bit=(best == "C_qlora_4bit")), "adapter")
sample = random.Random(1).sample(range(len(test)), 20)
again = predict(check, [test[i] for i in sample], batch_size=1)
same = sum(a["output"] == test_preds[i]["output"] for a, i in zip(again, sample))
print(f"{same}/20 regenerated predictions identical")
print(sorted(os.listdir("adapter")), f"{os.path.getsize('adapter/adapter_model.safetensors') / 2**20:.0f} MB")

In [ ]:
# Bundle everything that goes into the repository
with zipfile.ZipFile("submission_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["cleaning_log.csv", "predictions_test.jsonl", "predictions_dev.jsonl", "results.json"] + [f"adapter/{x}" for x in os.listdir("adapter")]:
        z.write(f)
    for k in runs: z.write(f"runs/{k}/dev_preds.json")
print(f"{os.path.getsize('submission_outputs.zip') / 2**20:.0f} MB")
try:
    from google.colab import files; files.download("submission_outputs.zip")
except Exception as e: print(e)